# Standard convex problems

Most convex problems we run into in graphics fall into one of five classes. They differ in the type of the objective and the type of the constraint set.

- **Linear program (LP)**: Linear objective, linear inequality constraints.

- **Quadratic program (QP)**: Convex quadratic objective, linear constraints.

- **Quadratically constrained quadratic program (QCQP)**: Convex quadratic objective and constraints.

- **Second-order cone program (SOCP)**: Linear objective, second-order cone constraints.

- **Semidefinite program (SDP)**: Linear objective, linear matrix inequality constraints.

These classes nest: every LP is also a QP, every QP is a QCQP, every QCQP is an SOCP, and every SOCP is an SDP. In practice we want to recognize the *smallest* family that fits our problem, because the more specialized the family, the more specialized the solver.

In this notebook we'll write a small CVXPY example for each of the five classes. We'll see two practical graphics applications, one for QP and one for SDP.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))
sys.path.append(str(pathlib.Path('../render_utility').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from pyplot_utils import plot_2d_function
from definitions import blue_2, red_2, orange_2, green_2, grey_1

## Linear program

$$ \min_x c^\top x \quad \text{subject to} \quad A x \le b. $$

The feasible set is a convex polytope and the optimum sits on a vertex. In notebook 7 we will see that the discrete Earth Mover's distance is exactly an LP.

In [ ]:
c = np.array([-2.0, -1.0])
A = np.array([[ 1.0,  1.0],
              [ 1.0, -1.0],
              [-1.0,  0.0],
              [ 0.0, -1.0]])
b = np.array([2.0, 2.0, 0.0, 0.0])

x = cp.Variable(2)
lp = cp.Problem(cp.Minimize(c @ x), [A @ x <= b])
lp.solve()
print('LP value:', lp.value)
print('LP x*:', x.value)

In [ ]:
def f_lp(x, y):
    return c[0] * x + c[1] * y

fig, ax = plt.subplots(figsize=(5.5, 5))
plt.sca(ax)
plot_2d_function(f_lp, xlim=(-0.5, 2.5), ylim=(-0.5, 2.5))

poly = np.array([[0, 0], [2, 0], [0, 2], [0, 0]])
ax.fill(poly[:, 0], poly[:, 1], color=grey_1, alpha=0.25)
ax.plot(poly[:, 0], poly[:, 1], color=grey_1, linewidth=5,
        solid_capstyle='round', solid_joinstyle='miter')
ax.scatter(*x.value, color=red_2, s=220, edgecolor='white', linewidth=2,
           zorder=6)
ax.annotate('optimum', xy=tuple(x.value), xytext=(10, 0),
            textcoords='offset points', va='center')

ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('LP on a polytope')
plt.tight_layout()
plt.show()

## Quadratic program

$$ \min_x \tfrac{1}{2} x^\top Q x + c^\top x \quad \text{subject to} \quad A x \le b, \quad Q \succeq 0. $$

QPs are extremely common in geometry processing because most least-squares problems with linear constraints land here. We will see one example in a moment with bounded biharmonic weights.

In [ ]:
Q = np.array([[2.0, 0.5],
              [0.5, 1.0]])
c = np.array([-3.0, -1.0])
x = cp.Variable(2)
qp = cp.Problem(cp.Minimize(0.5 * cp.quad_form(x, Q) + c @ x), [x >= -1, x <= 1])
qp.solve()
print('QP value:', qp.value)
print('QP x*:', x.value)

In [ ]:
def f_qp(x, y):
    return 0.5 * (Q[0, 0] * x**2 + 2 * Q[0, 1] * x * y + Q[1, 1] * y**2) \
        + c[0] * x + c[1] * y

fig, ax = plt.subplots(figsize=(5.5, 5))
plt.sca(ax)
plot_2d_function(f_qp, xlim=(-1.5, 2.0), ylim=(-1.5, 2.0))

box = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]])
ax.fill(box[:, 0], box[:, 1], color=grey_1, alpha=0.25)
ax.plot(box[:, 0], box[:, 1], color=grey_1, linewidth=5,
        solid_capstyle='round', solid_joinstyle='miter')

x_unc = -np.linalg.solve(Q, c)
ax.scatter(*x_unc, color=orange_2, marker='x', s=220, linewidths=3,
           zorder=6)
ax.annotate('unconstrained min', xy=tuple(x_unc), xytext=(10, 0),
            textcoords='offset points', va='center')
ax.scatter(*x.value, color=red_2, s=220, edgecolor='white', linewidth=2,
           zorder=6)
ax.annotate('constrained optimum', xy=tuple(x.value), xytext=(10, 0),
            textcoords='offset points', va='center')

ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('QP on a box')
plt.tight_layout()
plt.show()

## QP for bounded biharmonic weights

Bounded biharmonic weights by Jacobson, Baran, Popovic, Sorkine (2011) are the state-of-the-art for smooth shape deformation. For each handle $j$ we look for a smooth scalar field $w_j$ on the mesh that is one at handle $j$, zero at the other handles, between zero and one everywhere, and that sums to one across handles at every vertex:
$$ \min_{w_j} \tfrac{1}{2} \int_\Omega \|\Delta w_j\|^2 \, \mathrm{d}V \quad \text{subject to} \quad w_j|_{H_k} = \delta_{jk}, \ 0 \le w_j \le 1, \ \sum_j w_j = 1. $$
Discretized on a triangle mesh this is a QP with linear constraints. With those weights in hand, deforming the mesh is a single matrix-vector product per frame: each vertex follows a partition-of-unity weighted combination of the handle translations. This is called linear blend skinning.

**Interactive demo.** The cell below opens a polyscope window, where three red handles are placed on the bunny mesh. Click directly on one of the handles and drag to translate it: you'll see the bunny deforming in real-time. If you drag on empty space, the camera view still rotates. Once you're done, you can close the window to return to the notebook.

In [ ]:
from src.bbw_demo import bbw_demo

handle_positions = [[ 0.0,  0.5,  0.0],
                    [-0.4, -0.4,  0.3],
                    [ 0.4, -0.4, -0.3],]
bbw_demo('../data/bunny.obj', handle_positions=handle_positions)

## Quadratically constrained quadratic program

$$ \min_x \tfrac{1}{2} x^\top Q x + c^\top x \quad \text{subject to} \quad \tfrac{1}{2} x^\top P_i x + q_i^\top x + r_i \le 0, \quad Q, P_i \succeq 0. $$

When the $P_i$ are all PSD the feasible set is the intersection of ellipsoids. We will run into a *non-convex* QCQP in notebook 6, where we explore the Procrustes matching problem, and we will make the solve more efficient by relaxing it to an SDP.

In [ ]:
Q = np.eye(2)
c = np.array([-1.0, -1.0])
P1 = np.eye(2)
x = cp.Variable(2)
qcqp = cp.Problem(cp.Minimize(0.5 * cp.quad_form(x, Q) + c @ x),
                  [0.5 * cp.quad_form(x, P1) - 0.5 <= 0])
qcqp.solve()
print('QCQP value:', qcqp.value)
print('QCQP x*:', x.value)

In [ ]:
def f_qcqp(x, y):
    return 0.5 * (x**2 + y**2) + c[0] * x + c[1] * y

fig, ax = plt.subplots(figsize=(5.5, 5))
plt.sca(ax)
plot_2d_function(f_qcqp, xlim=(-1.3, 1.5), ylim=(-1.3, 1.5))

theta = np.linspace(0, 2 * np.pi, 200)
ax.fill(np.cos(theta), np.sin(theta), color=grey_1, alpha=0.25)
ax.plot(np.cos(theta), np.sin(theta), color=grey_1, linewidth=5,
        solid_capstyle='round', solid_joinstyle='miter')

x_unc = -c
ax.scatter(*x_unc, color=orange_2, marker='x', s=220, linewidths=3,
           zorder=6)
ax.annotate('unconstrained min', xy=tuple(x_unc), xytext=(10, 0),
            textcoords='offset points', va='center')
ax.scatter(*x.value, color=red_2, s=220, edgecolor='white', linewidth=2,
           zorder=6)
ax.annotate('constrained optimum', xy=tuple(x.value), xytext=(10, 0),
            textcoords='offset points', va='center')

ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('QCQP on a ball')
plt.tight_layout()
plt.show()

## Second-order cone program

$$ \min_x f^\top x \quad \text{subject to} \quad \|A_i x + b_i\|_2 \le c_i^\top x + d_i. $$

The cone constraint is more general than a quadratic one but a lot less general than a PSD one. SOCPs are a sweet spot in practice: expressive enough to cover most norms and most quadratic constraints, fast enough to solve at scale. We will use SOCP for the Benamou-Brenier formulation of optimal transport in notebook 7.

In [ ]:
f = np.array([1.0, 1.0])
A1 = np.eye(2)
b1 = np.zeros(2)
x = cp.Variable(2)
socp = cp.Problem(cp.Minimize(f @ x),
                  [cp.norm(A1 @ x + b1, 2) <= 1.0,
                   x >= -1])
socp.solve()
print('SOCP value:', socp.value)
print('SOCP x*:', x.value)

In [ ]:
def f_socp(x, y):
    return f[0] * x + f[1] * y

fig, ax = plt.subplots(figsize=(5.5, 5))
plt.sca(ax)
plot_2d_function(f_socp, xlim=(-1.3, 1.3), ylim=(-1.3, 1.3))

theta = np.linspace(0, 2 * np.pi, 200)
ax.fill(np.cos(theta), np.sin(theta), color=grey_1, alpha=0.25)
ax.plot(np.cos(theta), np.sin(theta), color=grey_1, linewidth=5,
        solid_capstyle='round', solid_joinstyle='miter')

ax.scatter(*x.value, color=red_2, s=220, edgecolor='white', linewidth=2,
           zorder=6)
ax.annotate('optimum', xy=tuple(x.value), xytext=(10, 0),
            textcoords='offset points', va='center')

ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('SOCP on a ball')
plt.tight_layout()
plt.show()

## Semidefinite program

$$ \min_X \operatorname{tr}(C^\top X) \quad \text{subject to} \quad \operatorname{tr}(A_i^\top X) \le b_i, \quad X \succeq 0. $$

SDPs are the most expressive convex family we will see. The variable is a symmetric matrix that has to be positive semidefinite. Two graphics applications we will meet are the singular value example below and Procrustes matching via the PM-SDP relaxation in notebook 6.

## SDP for bounded singular values

When we deform a mesh we often want to control how much each element stretches or compresses. Kovalsky, Aigerman, Basri, Lipman (2014) phrase this as a constraint on the singular values of the per-element deformation gradient $A \in \mathbb{R}^{n \times n}$. They want $\sigma_i(A) \in [\gamma, \Gamma]$ for given bounds $0 < \gamma \le \Gamma$. Both bounds can be written as linear matrix inequalities,
$$ \begin{pmatrix} \Gamma I & A \\ A^\top & \Gamma I \end{pmatrix} \succeq 0, \qquad \tfrac{1}{2}(A + A^\top) \succeq \gamma I. $$
The first bounds the largest singular value above by $\Gamma$, and the second bounds the smallest singular value below by $\gamma$. With these two LMIs in place, projecting a target Jacobian $B$ to the closest admissible $A$ becomes a small SDP per element.

We'll solve a single element so we can see the formulation in action!

In [ ]:
B = np.array([[1.5, 0.3], [0.0, 0.7]])
Gamma = 1.2
gamma = 0.8

A = cp.Variable((2, 2))
I = np.eye(2)
block = cp.bmat([[Gamma * I, A], [A.T, Gamma * I]])
constraints = [block >> 0, 0.5 * (A + A.T) - gamma * I >> 0]

sdp = cp.Problem(cp.Minimize(cp.norm(A - B, 'fro')), constraints)
sdp.solve()
print('SDP value:', sdp.value)
print('A*:')
print(A.value)
print('Singular values of A*:', np.linalg.svd(A.value, compute_uv=False))
print('Singular  values of B:', np.linalg.svd(B, compute_uv=False))

In [ ]:
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
B_def = B @ circle
A_def = A.value @ circle

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
for ax, deformed, title in zip(
    axes,
    [B_def, A_def],
    [f'Target $B$', f'Projected $A^*$']):
    ax.plot(circle[0], circle[1], color=grey_1, linewidth=2, linestyle='--',
            dash_capstyle='round')
    ax.annotate('unit circle', xy=(np.cos(np.pi / 4), np.sin(np.pi / 4)),
                xytext=(6, 6), textcoords='offset points')
    ax.plot(gamma * np.cos(theta), gamma * np.sin(theta), color=grey_1,
            linewidth=2, linestyle=':', dash_capstyle='round')
    ax.annotate(f'$\\gamma = {gamma}$',
                xy=(gamma * np.cos(-np.pi / 4), gamma * np.sin(-np.pi / 4)),
                xytext=(6, -10), textcoords='offset points')
    ax.plot(Gamma * np.cos(theta), Gamma * np.sin(theta), color=grey_1,
            linewidth=2, linestyle='-.', dash_capstyle='round')
    ax.annotate(f'$\\Gamma = {Gamma}$',
                xy=(Gamma * np.cos(np.pi / 3), Gamma * np.sin(np.pi / 3)),
                xytext=(6, 6), textcoords='offset points')
    ax.plot(deformed[0], deformed[1], color=blue_2, linewidth=5,
            solid_capstyle='round', solid_joinstyle='miter')
    ax.annotate('image', xy=(deformed[0, 0], deformed[1, 0]),
                xytext=(8, 0), textcoords='offset points', color=blue_2)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

The singular values of the projected $A^*$ now sit inside $[\gamma, \Gamma]$, while $B$ violates the upper bound. The same per-element projection is what gives the paper's strong distortion guarantees on full meshes.

## Summary

Each of the five families maps cleanly to a graphics application we will see later in the tutorial. The common thread is that as soon as we recognize a problem as a member of one of these families, a solver is one CVXPY call away.

## References

[1] *Convex Optimization*. Boyd, S.; Vandenberghe, L.
       Cambridge University Press (2004).
       :DOI:`10.1017/CBO9780511804441`

[2] *Bounded biharmonic weights for real-time deformation*.
       Jacobson, A.; Baran, I.; Popovic, J.; Sorkine, O.
       ACM Transactions on Graphics (SIGGRAPH) 30 (4): 78 (2011).
       :DOI:`10.1145/2010324.1964973`

[3] *Controlling singular values with semidefinite programming*.
       Kovalsky, S.; Aigerman, N.; Basri, R.; Lipman, Y.
       ACM Transactions on Graphics (SIGGRAPH) 33 (4): 68 (2014).
       :DOI:`10.1145/2601097.2601142`